<a href="https://colab.research.google.com/github/MariuszMleczak/Projekt_AI_-_Klasyfikator_defektow_powierzchni_stalowych/blob/main/trening_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/AI_PROJECT/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database

Dataset URL: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
License(s): unknown
100% 26.4M/26.4M [00:00<00:00, 119MB/s]



In [ ]:
!unzip -q neu-surface-defect-database.zip -d /content/NEU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
!find /content/NEU -type d

/content/NEU
/content/NEU/NEU-DET
/content/NEU/NEU-DET/validation
/content/NEU/NEU-DET/validation/images
/content/NEU/NEU-DET/validation/images/scratches
/content/NEU/NEU-DET/validation/images/crazing
/content/NEU/NEU-DET/validation/images/patches
/content/NEU/NEU-DET/validation/images/rolled-in_scale
/content/NEU/NEU-DET/validation/images/pitted_surface
/content/NEU/NEU-DET/validation/images/inclusion
/content/NEU/NEU-DET/validation/annotations
/content/NEU/NEU-DET/train
/content/NEU/NEU-DET/train/images
/content/NEU/NEU-DET/train/images/scratches
/content/NEU/NEU-DET/train/images/crazing
/content/NEU/NEU-DET/train/images/patches
/content/NEU/NEU-DET/train/images/rolled-in_scale
/content/NEU/NEU-DET/train/images/pitted_surface
/content/NEU/NEU-DET/train/images/inclusion
/content/NEU/NEU-DET/train/annotations


In [ ]:
BASE_PATH = "/content/NEU/NEU-DET"

train_dataset = datasets.ImageFolder(
    f"{BASE_PATH}/train/images",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    f"{BASE_PATH}/validation/images",
    transform=transform
)

print(train_dataset.classes)
print(len(train_dataset.classes))

['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
6


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2, pin_memory=True)

In [ ]:
weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 6)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 169MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss = {total_loss/len(train_loader)}")

Epoch 1: loss = 0.9086678167184193
Epoch 2: loss = 0.3037337710460027
Epoch 3: loss = 0.19769261644946204
Epoch 4: loss = 0.14754908291829957
Epoch 5: loss = 0.1180074227352937


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", correct / total)

Accuracy: 0.9527777777777777


In [ ]:
torch.save(model.state_dict(), "neu_model.pth")
print("model saved")

model saved


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_path = "/content/drive/MyDrive/AI_PROJECT/Wytrenowany_model_AI/neu_model.pth"

torch.save(model.state_dict(), save_path)

print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/AI_PROJECT/Wytrenowany_model_AI/neu_model.pth


In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/AI_PROJECT/Wytrenowany_model_AI/neu_model.pth"))

True
